<a href="https://colab.research.google.com/github/hye0-n0/KRX_stock_prediction/blob/main/%5B0711%5DMonte_Carlo_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 라이브러리 다운로드
!pip install finance-datareader #주식 관련 라이브러리
!pip install PublicDataReader #한국 경제 통계 라이브러리

In [ ]:
import random
import os
import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
from itertools import repeat
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import tensorflow.keras as keras
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, RNN, Dropout, Flatten
from keras.optimizers import SGD
from sklearn.impute import SimpleImputer
from joblib import Parallel, delayed
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42) # Seed 고정

## DataLoad

공유드라이브는 mounting이 힘들기 때문에 </br>공유함에 있는 데이터 파일 open의 shortcut을 nyDrive에 만들면 됩니다

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/open/train.csv')
df

### 1. **데이터 셋 조합하기**

- Monte Carlo에는 굳이 필요없지만 공통부문이므로 놔둠

In [ ]:
def fetch_stock_data(code, start_date, end_date, base_rate):
    try:
        stock_data = fdr.DataReader(code, start=start_date, end=end_date)
        stock_data.reset_index(inplace=True)
        stock_data['StockCode'] = code
        stock_data['Date'] = pd.to_datetime(stock_data['Date']).dt.strftime('%Y-%m-%d')
        base_rate['시점'] = pd.to_datetime(base_rate['시점']).dt.strftime('%Y-%m-%d')
        stock_data = stock_data.merge(base_rate, left_on='Date', right_on='시점', how='left')
        stock_data = stock_data.drop(["시점"], axis=1)
        stock_data.rename(columns={'값': '금리'}, inplace=True)
        return stock_data
    except Exception as e:
        print(f"Error fetching data for {code}: {e}")


- 기준 금리 가져오기

In [ ]:
service_key = "AM3PK2JF40CDOUQX5NJ9"
api = Ecos(service_key)

start_date = '20200529'
end_date = '20230530'

base_rate = api.get_statistic_search(통계표코드="722Y001", 주기="D", 검색시작일자=start_date, 검색종료일자=end_date)
base_rate = base_rate[base_rate['통계항목명1'] == '한국은행 기준금리'][['시점', '값']]

original_stock_codes = df['종목코드'].unique()
stock_codes = [code[1:] for code in original_stock_codes]

with ThreadPoolExecutor() as executor:
    data_frames = list(tqdm(executor.map(fetch_stock_data, stock_codes, repeat(start_date), repeat(end_date), repeat(base_rate)), total=len(stock_codes)))

filtered_data_frames = [df for df in data_frames if df is not None]

merged_df = pd.concat(filtered_data_frames, axis=0)
df = merged_df.sort_values(by=['StockCode', merged_df.columns[0]]).reset_index(drop=True)

cols = list(df.columns)
cols.remove('StockCode')
new_cols_order = [cols[0], 'StockCode'] + cols[1:]
df = df[new_cols_order]

df.to_csv('stock_data_with_interest_rate.csv', index=False)
df

- **이미 처리해놓은 데이터 가져와서 쓰기** </br>
공유드라이브 data에 있는 파일 각자 경로 지정해서 쓰시면 됩니다


In [ ]:
# 그냥 불러와서 사용하는 것도 GOOD choice
df = pd.read_csv('/content/drive/MyDrive/busy_kids/dacon_krx/data/stock_data_with_interest_rate.csv', dtype={'StockCode': str})
df

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/final_data.csv')
df['StockCode'] = df['StockCode'].str.replace('A', '')
df

In [ ]:
# 결측치가 있는 행의 인덱스를 가져옵니다.
null_data_index = df.index[df['Change'].isnull()].tolist()

for i in null_data_index:
    prev_close = df.loc[i-1, 'Close']
    curr_close = df.loc[i, 'Close']
    change = (curr_close - prev_close) / prev_close
    df.loc[i, 'Change'] = change

df.isnull().sum()

### (접어두고 쓰세요) 종목 한개로 진행해 경우의 수 분포 확인하기

In [ ]:
returns = np.log(1 + single_stock_df['Close'].pct_change())
mu, sigma = returns.mean(), returns.std()
initial = single_df['Close'].iloc[-1]

sim_rets = np.random.normal(mu, sigma, 15)
sim_prices = initial * (sim_rets +1).cumprod()
last_prices.append(sim_prices[-1])

In [ ]:
print("Initial Prices:", initial)
print("Expected price: ", round(np.mean(last_prices),2))
print("Quantile (5%): ",np.percentile(last_prices,5))
print("Quantile (95%): ",np.percentile(last_prices,95))

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(last_prices, bins=100)
plt.axvline(np.percentile(last_prices,5), color='r', linestyle='dashed', linewidth=2)
plt.axvline(np.percentile(last_prices,95), color='r', linestyle='dashed', linewidth=2)
plt.show()

### 2. **전체 종목에 대한 Simulation**

In [ ]:
stock_codes = []
final_returns = []

for i in tqdm(df['StockCode'].unique(), desc="Simulating"):
  single_stock_df = df[df['StockCode']== i]
  returns = np.log(1 + single_stock_df['Close'].pct_change())
  mu, sigma = returns.mean(), returns.std()
  initial = single_stock_df['Close'].iloc[-1]

  last_prices = []

  for t in range(1000):
    sim_rets = np.random.normal(mu, sigma, 15)
    sim_prices = initial * (sim_rets +1).cumprod()
    last_prices.append(sim_prices[-1])

  final_price = np.mean(last_prices)
  final_return = (final_price - initial)/initial

  stock_codes.append(i)
  final_returns.append(final_return)

result_df = pd.DataFrame({'StockCode': stock_codes, 'Final_return': final_returns})


In [ ]:
results_df = result_df.copy()

### 3. **순위 매겨서 제출하기**

In [ ]:
results_df['StockCode'] = 'A' + results_df['StockCode'].astype(str)
results_df['순위'] = results_df['Final_return'].rank(method='first', ascending=False).astype('int')
results_df = results_df.rename(columns={'StockCode': '종목코드'})
results_df

In [ ]:
sample_submission = pd.read_csv('/content/drive/MyDrive/open/sample_submission.csv')
sample_submission

In [ ]:
baseline_submission = sample_submission[['종목코드']].merge(results_df[['종목코드', '순위']], on='종목코드', how='left')
baseline_submission

In [ ]:
baseline_submission.to_csv('Monte_Carlo_short_submission2.csv', index=False)